# Energy Demand Forecasting — Probabilistic Predictions

**Dataset:** PJM East (PJME) Hourly Energy Consumption 2001–2018  
**Goal:** Forecast energy demand with prediction intervals using quantile regression

**Models:**
- XGBoost Quantile Regression (baseline)
- LightGBM Quantile Regression (improved)

**Evaluation:** Pinball loss, prediction interval coverage, interval width

In [ ]:
!pip install xgboost lightgbm holidays openmeteo-requests requests-cache retry-requests -q
print('Done.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import holidays
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')

print('Libraries loaded.')

## 1. Load Data

In [ ]:
import os

csv_path = 'PJME_hourly.csv' if os.path.exists('PJME_hourly.csv') else 'data/PJME_hourly.csv'

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
date_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
df[date_col] = pd.to_datetime(df[date_col])
df = df.set_index(date_col).sort_index()
df.columns = ['demand_mw']
df.index.name = 'Datetime'

print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
df.head()

## 2. Feature Engineering

We build features from three sources:
- **Calendar features**: hour, day of week, month, weekend, holidays
- **Lag features**: demand 1h, 24h, 168h (1 week) ago
- **Rolling statistics**: 24h and 168h rolling mean and std

In [ ]:
us_holidays = holidays.US(years=range(2001, 2019))

def build_features(df):
    d = df.copy()

    # Calendar
    d['hour']       = d.index.hour
    d['dayofweek']  = d.index.dayofweek
    d['month']      = d.index.month
    d['quarter']    = d.index.quarter
    d['dayofyear']  = d.index.dayofyear
    d['weekofyear'] = d.index.isocalendar().week.astype(int)
    d['is_weekend'] = d['dayofweek'].isin([5, 6]).astype(int)
    d['is_holiday'] = d.index.normalize().isin(us_holidays).astype(int)

    # Cyclical encoding for hour and month (captures periodicity)
    d['hour_sin']   = np.sin(2 * np.pi * d['hour'] / 24)
    d['hour_cos']   = np.cos(2 * np.pi * d['hour'] / 24)
    d['month_sin']  = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos']  = np.cos(2 * np.pi * d['month'] / 12)
    d['dow_sin']    = np.sin(2 * np.pi * d['dayofweek'] / 7)
    d['dow_cos']    = np.cos(2 * np.pi * d['dayofweek'] / 7)

    # Lag features
    d['lag_1h']   = d['demand_mw'].shift(1)
    d['lag_24h']  = d['demand_mw'].shift(24)
    d['lag_168h'] = d['demand_mw'].shift(168)

    # Rolling stats (based on lag to avoid leakage)
    d['roll_mean_24h']  = d['demand_mw'].shift(1).rolling(24).mean()
    d['roll_std_24h']   = d['demand_mw'].shift(1).rolling(24).std()
    d['roll_mean_168h'] = d['demand_mw'].shift(1).rolling(168).mean()
    d['roll_std_168h']  = d['demand_mw'].shift(1).rolling(168).std()

    return d.dropna()

df_feat = build_features(df)
print(f'Features shape: {df_feat.shape}')
print(f'Feature columns: {[c for c in df_feat.columns if c != "demand_mw"]}')
df_feat.head(3)

## 3. Train / Test Split

We use the last full year (2018) as the test set. This is a **time-based split** — we never train on future data.

In [ ]:
FEATURE_COLS = [c for c in df_feat.columns if c != 'demand_mw']
TARGET = 'demand_mw'

train = df_feat[df_feat.index.year < 2018]
test  = df_feat[df_feat.index.year == 2018]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET]

print(f'Train: {train.index.min().date()} → {train.index.max().date()}  ({len(train):,} rows)')
print(f'Test:  {test.index.min().date()}  → {test.index.max().date()}   ({len(test):,} rows)')

# Visualise the split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, y_train, color='steelblue', linewidth=0.4, label='Train')
ax.plot(test.index,  y_test,  color='tomato',    linewidth=0.6, label='Test (2018)')
ax.set_title('Train / Test Split', fontsize=13, fontweight='bold')
ax.set_ylabel('Demand (MW)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Quantile Regression — What and Why?

Standard regression predicts a single value (the mean). **Quantile regression** predicts multiple quantiles simultaneously:
- **Q10** (10th percentile) — lower bound: demand is unlikely to fall below this
- **Q50** (50th percentile) — median forecast: the best single-point estimate  
- **Q90** (90th percentile) — upper bound: demand is unlikely to exceed this

Together, Q10–Q90 form an **80% prediction interval** — a range we expect to contain the true demand 80% of the time. This is much more useful for grid operators than a single point forecast.

## 5. Model 1 — XGBoost Quantile Regression

In [ ]:
QUANTILES = [0.1, 0.5, 0.9]

xgb_preds = {}
for q in QUANTILES:
    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:quantileerror',
        quantile_alpha=q,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    xgb_preds[q] = model.predict(X_test)
    print(f'  Q{int(q*100):02d} trained.')

xgb_df = pd.DataFrame(xgb_preds, index=test.index)
xgb_df.columns = ['xgb_q10', 'xgb_q50', 'xgb_q90']
xgb_df['actual'] = y_test.values
print('\nXGBoost predictions sample:')
xgb_df.head()

## 6. Model 2 — LightGBM Quantile Regression

In [ ]:
lgbm_preds = {}
for q in QUANTILES:
    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='quantile',
        alpha=q,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(X_train, y_train)
    lgbm_preds[q] = model.predict(X_test)
    print(f'  Q{int(q*100):02d} trained.')

lgbm_df = pd.DataFrame(lgbm_preds, index=test.index)
lgbm_df.columns = ['lgbm_q10', 'lgbm_q50', 'lgbm_q90']
lgbm_df['actual'] = y_test.values
print('\nLightGBM predictions sample:')
lgbm_df.head()

## 7. Evaluation

We evaluate using three metrics:

- **MAE** (Mean Absolute Error) on Q50 — point forecast accuracy
- **Pinball Loss** — the proper scoring rule for quantile forecasts; lower is better
- **Coverage** — what % of actual values fall inside the Q10–Q90 interval (target: ~80%)
- **Interval Width** — average width of the prediction interval; narrower = more precise

In [ ]:
def pinball_loss(y_true, y_pred, q):
    e = y_true - y_pred
    return np.mean(np.where(e >= 0, q * e, (q - 1) * e))

def evaluate(name, actual, q10, q50, q90):
    mae      = mean_absolute_error(actual, q50)
    pb10     = pinball_loss(actual, q10, 0.1)
    pb50     = pinball_loss(actual, q50, 0.5)
    pb90     = pinball_loss(actual, q90, 0.9)
    avg_pb   = (pb10 + pb50 + pb90) / 3
    coverage = np.mean((actual >= q10) & (actual <= q90)) * 100
    width    = np.mean(q90 - q10)
    print(f'\n=== {name} ===')
    print(f'  MAE (Q50):        {mae:,.1f} MW')
    print(f'  Pinball Q10:      {pb10:,.1f}')
    print(f'  Pinball Q50:      {pb50:,.1f}')
    print(f'  Pinball Q90:      {pb90:,.1f}')
    print(f'  Avg Pinball Loss: {avg_pb:,.1f}')
    print(f'  Coverage (80%):   {coverage:.1f}%  (target ≈ 80%)')
    print(f'  Interval Width:   {width:,.1f} MW')
    return {'model': name, 'MAE': mae, 'Avg Pinball': avg_pb,
            'Coverage %': coverage, 'Interval Width': width}

results = []
results.append(evaluate('XGBoost',  y_test.values,
                         xgb_df['xgb_q10'].values,
                         xgb_df['xgb_q50'].values,
                         xgb_df['xgb_q90'].values))
results.append(evaluate('LightGBM', y_test.values,
                         lgbm_df['lgbm_q10'].values,
                         lgbm_df['lgbm_q50'].values,
                         lgbm_df['lgbm_q90'].values))

results_df = pd.DataFrame(results).set_index('model').round(1)
print('\n\n=== Summary Table ===')
print(results_df)

## 8. Visualisations

### 8a. Prediction Intervals — Full Test Period

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

for ax, name, df_pred, q10_col, q50_col, q90_col in [
    (axes[0], 'XGBoost',  xgb_df,  'xgb_q10',  'xgb_q50',  'xgb_q90'),
    (axes[1], 'LightGBM', lgbm_df, 'lgbm_q10', 'lgbm_q50', 'lgbm_q90'),
]:
    ax.fill_between(df_pred.index, df_pred[q10_col], df_pred[q90_col],
                    alpha=0.25, color='steelblue', label='80% interval (Q10–Q90)')
    ax.plot(df_pred.index, df_pred[q50_col], color='steelblue',
            linewidth=0.8, label='Q50 forecast', alpha=0.9)
    ax.plot(df_pred.index, df_pred['actual'], color='tomato',
            linewidth=0.5, alpha=0.7, label='Actual demand')
    ax.set_title(f'{name} — 2018 Forecast with Prediction Intervals', fontweight='bold')
    ax.set_ylabel('Demand (MW)')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

### 8b. Zoom — One Week in January (Winter Peak)

In [ ]:
zoom_start = '2018-01-08'
zoom_end   = '2018-01-14'

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ax, name, df_pred, q10_col, q50_col, q90_col in [
    (axes[0], 'XGBoost',  xgb_df,  'xgb_q10',  'xgb_q50',  'xgb_q90'),
    (axes[1], 'LightGBM', lgbm_df, 'lgbm_q10', 'lgbm_q50', 'lgbm_q90'),
]:
    z = df_pred.loc[zoom_start:zoom_end]
    ax.fill_between(z.index, z[q10_col], z[q90_col],
                    alpha=0.3, color='steelblue', label='80% interval')
    ax.plot(z.index, z[q50_col], color='steelblue', linewidth=2, label='Q50 forecast')
    ax.plot(z.index, z['actual'], color='tomato', linewidth=2, linestyle='--', label='Actual')
    ax.set_title(f'{name} — Week of Jan 8–14 2018 (Winter Peak)', fontweight='bold')
    ax.set_ylabel('Demand (MW)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d'))
    ax.legend()

plt.tight_layout()
plt.show()

### 8c. Feature Importance (LightGBM)

In [ ]:
# Retrain LightGBM Q50 to extract feature importances
lgbm_q50 = LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    objective='quantile', alpha=0.5,
    random_state=42, n_jobs=-1, verbose=-1
)
lgbm_q50.fit(X_train, y_train)

importance = pd.Series(lgbm_q50.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
top20 = importance.tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('LightGBM Feature Importance (Top 20)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 9. Reflection

### What worked
- Lag features (especially `lag_24h` and `lag_168h`) were the strongest predictors — energy demand is highly autocorrelated
- LightGBM trained significantly faster than XGBoost and produced tighter prediction intervals
- Calendar features (hour, day of week, month) captured the strong seasonality patterns identified in EDA
- Cyclical encoding (sin/cos) for hour and month helped the model understand periodicity

### What did not work / limitations
- The weather API data was only fetched for 2017, not integrated into the forecasting features — adding real temperature data would likely improve accuracy
- The models assume future lag values are known, which works for 1-step-ahead forecasting but not for longer horizons without recursive prediction
- Extreme demand events (heat waves, polar vortex) are hard to predict without real-time weather

### What I learned
- **Quantile regression** is more useful than point forecasts for grid operators — knowing the uncertainty range helps with reserve planning
- **Pinball loss** is the right metric for quantile forecasts, not RMSE or MAE alone
- **Coverage** tells you if your prediction intervals are well-calibrated — if 80% of actual values fall inside the Q10–Q90 band, the model is well-calibrated
- Lag features from the same hour last week (`lag_168h`) are extremely powerful for weekly-seasonal data like energy